In [1]:
import csv
import pandas as pd

# Ler o arquivo
linhas = []
with open("tabela_1849_industria.csv", "r", encoding="utf-8-sig") as f:
    reader = csv.reader(f, delimiter=";")
    for row in reader:
        linhas.append(row)

# Encontrar onde começa cada variável
variaveis = {}
for i, row in enumerate(linhas):
    if row and row[0].startswith("Variável - "):
        nome_var = row[0].replace("Variável - ", "").strip()
        variaveis[i] = nome_var

print(f"Variáveis encontradas: {len(variaveis)}")
for idx, nome in variaveis.items():
    print(f"  Linha {idx}: {nome}")

# Extrair dados
registros = []

for idx_var, nome_var in variaveis.items():
    # Os dados começam 2 linhas depois do título da variável (pula cabeçalho)
    i = idx_var + 2
    while i < len(linhas):
        row = linhas[i]

        # Se chegou na próxima variável, para
        if row and row[0].startswith("Variável - "):
            break

        # Se for linha de dados (começa com "UF")
        if row and row[0].startswith("UF"):
            # Parsear a string interna (split por vírgula, respeitando aspas)
            parser = csv.reader([row[0]], skipinitialspace=True)
            partes = list(parser)[0]

            # partes = ['UF', '11', 'Rondônia', '2007', 'Total', '1125', 'Unidades']
            if len(partes) >= 7:
                uf = partes[2]
                cod = partes[1]
                ano = partes[3]
                setor = partes[4]
                valor_str = partes[5]

                # Converter valor para número
                try:
                    valor = float(valor_str)
                except:
                    valor = None

                registros.append({
                    "UF": uf,
                    "Cód_IBGE": cod,
                    "Ano": int(ano),
                    "Setor": setor,
                    "Valor": valor,
                    "Variável": nome_var
                })

        # Se encontrar linha de rodapé (Fonte), para
        if row and "Fonte:" in row[0]:
            break

        i += 1

# Criar DataFrame
df_final = pd.DataFrame(registros)

print(f"\n🎉 SHAPE FINAL: {df_final.shape}")
print(f"📅 Anos: {sorted(df_final['Ano'].unique())}")
print(f"🏭 Setores: {sorted(df_final['Setor'].unique())}")
print(f"🗺️  UFs: {df_final['UF'].nunique()} estados")
print(f"📊 Variáveis: {df_final['Variável'].nunique()}")
print(f"\nPrimeiras 10 linhas:")
df_final.head(10)

Variáveis encontradas: 9
  Linha 1: Número de unidades locais
  Linha 3219: Pessoal ocupado em 31/12
  Linha 6437: Salários, retiradas e outras remunerações
  Linha 9655: Total de receitas líquidas de vendas
  Linha 12873: Receita líquida de vendas de atividades industriais
  Linha 16091: Total de custos e despesas
  Linha 19309: Custos com consumo de matérias-primas, materiais auxiliares e componentes
  Linha 22527: Valor bruto da produção industrial
  Linha 25745: Valor da transformação industrial

🎉 SHAPE FINAL: (28917, 6)
📅 Anos: [np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
🏭 Setores: ['10 Fabricação de produtos alimentícios', '20 Fabricação de produtos químicos', '24 Metalurgia', '29 Fabricação de veículos automotores, reboques e carrocerias', 'B Indústrias

,UF,Cód_IBGE,Ano,Setor,Valor,Variável
0,Rondônia,11,2007,Total,1125.0,Número de unidades locais
1,Rondônia,11,2007,B Indústrias extrativas,29.0,Número de unidades locais
2,Rondônia,11,2007,C Indústrias de transformação,1096.0,Número de unidades locais
3,Rondônia,11,2007,10 Fabricação de produtos alimentícios,207.0,Número de unidades locais
4,Rondônia,11,2007,20 Fabricação de produtos químicos,12.0,Número de unidades locais
5,Rondônia,11,2007,24 Metalurgia,11.0,Número de unidades locais
6,Rondônia,11,2007,"29 Fabricação de veículos automotores, reboque...",28.0,Número de unidades locais
7,Rondônia,11,2008,Total,1164.0,Número de unidades locais
8,Rondônia,11,2008,B Indústrias extrativas,37.0,Número de unidades locais
9,Rondônia,11,2008,C Indústrias de transformação,1127.0,Número de unidades locais


In [2]:
# Salvar CSV limpo
df_final.to_csv("industria_limpa.csv", index=False, encoding="utf-8")
print("💾 Salvo como 'industria_limpa.csv' na pasta Desktop!")

💾 Salvo como 'industria_limpa.csv' na pasta Desktop!


In [4]:
import sqlite3
import pandas as pd

# Define the database path for Colab environment
db_path = "/content/industria_brasileira.db"

# Connect to the database (it will create the file if it doesn't exist)
conn = sqlite3.connect(db_path)

# Load the cleaned CSV into a DataFrame
df_cleaned = pd.read_csv("/content/industria_limpa.csv")

# Save the DataFrame to a SQL table named 'industria'
df_cleaned.to_sql("industria", conn, if_exists="replace", index=False)

print(f"✅ Database '{db_path}' created and populated with data from 'industria_limpa.csv'.")

# ============================================
# CSV 1: Top estados 2023 (Query 1)
# ============================================
df1 = pd.read_sql("""
    SELECT UF, Valor AS Unidades
    FROM industria
    WHERE Variável = 'Número de unidades locais'
      AND Setor = 'Total'
      AND Ano = 2023
    ORDER BY Valor DESC
""", conn)
df1.to_csv("grafico1_top_estados.csv", index=False)
print(f"✅ CSV 1: {len(df1)} estados")

# ============================================
# CSV 2: Evolução empresas (Query 2)
# ============================================
df2 = pd.read_sql("""
    SELECT Ano, SUM(Valor) AS Total_Unidades
    FROM industria
    WHERE Variável = 'Número de unidades locais'
      AND Setor = 'Total'
    GROUP BY Ano
    ORDER BY Ano
""", conn)
df2.to_csv("grafico2_evolucao_empresas.csv", index=False)
print(f"✅ CSV 2: {len(df2)} anos")

# ============================================
# CSV 3: Salário médio mensal (Query 4)
# ============================================
df3 = pd.read_sql("""
    SELECT s.Ano,
           ROUND(SUM(s.Valor) / NULLIF(SUM(p.Valor), 0) / 12, 2) AS Salario_Medio_Mensal
    FROM industria s
    JOIN industria p
        ON s.UF = p.UF AND s.Ano = p.Ano AND s.Setor = p.Setor
    WHERE s.Variável = 'Salários, retiradas e outras remunerações'
      AND p.Variável = 'Pessoal ocupado em 31/12'
      AND s.Setor = 'Total'
    GROUP BY s.Ano
    ORDER BY s.Ano
""", conn)
df3.to_csv("grafico3_salario_medio.csv", index=False)
print(f"✅ CSV 3: {len(df3)} anos")

# ============================================
# CSV 4: Receita vs Custos (Query 5)
# ============================================
df4 = pd.read_sql("""
    SELECT r.Ano,
           ROUND(SUM(r.Valor) / 1000000, 2) AS Receita_Bilhoes,
           ROUND(SUM(c.Valor) / 1000000, 2) AS Custos_Bilhoes,
           ROUND((SUM(r.Valor) - SUM(c.Valor)) / 1000000, 2) AS Margem_Bilhoes
    FROM industria r
    JOIN industria c
        ON r.UF = c.UF AND r.Ano = c.Ano AND r.Setor = c.Setor
    WHERE r.Variável = 'Total de receitas líquidas de vendas'
      AND c.Variável = 'Total de custos e despesas'
      AND r.Setor = 'Total'
    GROUP BY r.Ano
    ORDER BY r.Ano
""", conn)
df4.to_csv("grafico4_receita_custos.csv", index=False)
print(f"✅ CSV 4: {len(df4)} anos")

# ============================================
# CSV 5: Setores 2023 (Query 6)
# ============================================
df5 = pd.read_sql("""
    SELECT Setor,
           ROUND(SUM(CASE WHEN Variável = 'Valor da transformação industrial' THEN Valor ELSE 0 END) / 1000000, 2) AS VTI_Bilhoes,
           ROUND(SUM(CASE WHEN Variável = 'Pessoal ocupado em 31/12' THEN Valor ELSE 0 END), 0) AS Pessoas,
           ROUND(SUM(CASE WHEN Variável = 'Valor da transformação industrial' THEN Valor ELSE 0 END) /
                 NULLIF(SUM(CASE WHEN Variável = 'Pessoal ocupado em 31/12' THEN Valor ELSE 0 END), 0), 2) AS Valor_Por_Trabalhador
    FROM industria
    WHERE Ano = 2023
    GROUP BY Setor
    ORDER BY VTI_Bilhoes DESC
""", conn)
df5.to_csv("grafico5_setores.csv", index=False)
print(f"✅ CSV 5: {len(df5)} setores")

# ============================================
# CSV 6: Mapa Brasil (todos estados, 2023)
# ============================================
df6 = pd.read_sql("""
    SELECT UF,
           SUM(CASE WHEN Variável = 'Número de unidades locais' AND Setor = 'Total' THEN Valor ELSE 0 END) AS Unidades,
           SUM(CASE WHEN Variável = 'Pessoal ocupado em 31/12' AND Setor = 'Total' THEN Valor ELSE 0 END) AS Pessoas,
           SUM(CASE WHEN Variável = 'Valor da transformação industrial' AND Setor = 'Total' THEN Valor ELSE 0 END) / 1000000 AS VTI_Bilhoes
    FROM industria
    WHERE Ano = 2023
    GROUP BY UF
""", conn)
df6.to_csv("grafico6_mapa_brasil.csv", index=False)
print(f"✅ CSV 6: {len(df6)} estados")

conn.close()
print(f"\n🎉 Todos os CSVs gerados na pasta do Colab (e.g., /content/)!")


✅ Database '/content/industria_brasileira.db' created and populated with data from 'industria_limpa.csv'.
✅ CSV 1: 27 estados
✅ CSV 2: 17 anos
✅ CSV 3: 17 anos
✅ CSV 4: 17 anos
✅ CSV 5: 7 setores
✅ CSV 6: 27 estados

🎉 Todos os CSVs gerados na pasta do Colab (e.g., /content/)!


In [5]:
import pandas as pd

df = pd.read_csv("grafico2_evolucao_empresas.csv")
print(df.head())
print(f"\nShape: {df.shape}")
print(f"Colunas: {df.columns.tolist()}")

    Ano  Total_Unidades
0  2007        172681.0
1  2008        182153.0
2  2009        185577.0
3  2010        189530.0
4  2011        198941.0

Shape: (17, 2)
Colunas: ['Ano', 'Total_Unidades']
